# 04 — Reject Inference: Exploring Selection Bias

When a lender only observes outcomes for approved applicants, the training data may suffer from **selection bias**.  
Rejected applicants — whose outcomes are unobserved — are invisible to supervised training and evaluation.

**Approach: Parcelling (Simple Augmentation)**  
1. Load the accepted-only champion  
2. Score rejected applicants with it  
3. Assign hard pseudo-labels by score rank  
4. Retrain on accepted and down-weighted pseudo-labeled data

The notebook calls the production reject-inference helpers so its behavior stays aligned with the pipeline. Parcelling remains a heuristic and cannot prove improvement on rejected applicants without observed outcomes.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve


def find_repo_root():
    """Locate the repository whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pipeline").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the GBM_cs repository")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline import reject_inference  # noqa: E402

plt.style.use("seaborn-v0_8-whitegrid")

train, val, test, rejected, accepted_model, feature_cols = reject_inference.load_data()

print(f"Accepted train: {len(train):,} rows")
print(f"Rejected: {len(rejected):,} rows")
print(f"\nAccepted features: {feature_cols[:5]}...")
print(f"Rejected features: {rejected.columns.tolist()}")

## 1. Align Rejected Data to Accepted Feature Space

Rejected applicants have fewer columns. We use the overlap (FICO, DTI, loan amount, emp_length) and fill the rest with population medians from the accepted training set.

In [ ]:
# The helper samples the configured cap, maps every available overlapping
# feature, fills unavailable values from training medians, and fixes column order.
rejected_aligned = reject_inference.align_rejected_features(
    rejected, feature_cols, train
)

print(f"Rejected aligned: {rejected_aligned.shape}")
print(f"Null check: {rejected_aligned.isnull().sum().sum()} nulls")

## 2. Score Rejected Applicants & Assign Pseudo-Labels

In [ ]:
training_default_rate = train["default"].mean()
pseudo_labels, rejected_scores = reject_inference.assign_pseudo_labels(
    accepted_model, rejected_aligned, training_default_rate
)
rejected_aligned = rejected_aligned.copy()
rejected_aligned["default"] = pseudo_labels
threshold = float(rejected_scores[pseudo_labels == 1].min())

print("Pseudo-label distribution:")
print(f"  Non-default: {(pseudo_labels == 0).sum():,}")
print(f"  Default:     {(pseudo_labels == 1).sum():,}")
print(f"  Rate:        {pseudo_labels.mean():.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    accepted_model.predict_proba(train[feature_cols])[:, 1],
    bins=50,
    alpha=0.5,
    density=True,
    label="Accepted (known outcomes)",
    color="#2ecc71",
)
ax.hist(
    rejected_scores,
    bins=50,
    alpha=0.5,
    density=True,
    label="Rejected (pseudo-labels)",
    color="#e74c3c",
)
ax.axvline(
    threshold,
    color="black",
    linestyle="--",
    label=f"Pseudo-label threshold ({threshold:.3f})",
)
ax.set_xlabel("Raw Model Score")
ax.set_ylabel("Density")
ax.set_title("Score Distribution: Accepted vs Rejected Applicants")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Retrain on Combined Dataset & Compare

In [ ]:
# Reuse production LightGBM parameters, sample weights, and early-stopping split.
# The returned calibration carve-out excludes every fabricated outcome.
augmented_model, X_calibration, y_calibration = (
    reject_inference.train_augmented_model(
        train[feature_cols],
        train["default"],
        rejected_aligned[feature_cols],
        rejected_aligned["default"],
        val[feature_cols],
        val["default"],
    )
)

print(f"Augmented model best iteration: {augmented_model.best_iteration_}")
print(f"Observed-only calibration rows: {len(X_calibration):,}")

## 4. Compare: Accepted-Only vs Augmented Model

In [ ]:
X_test = test[feature_cols]
y_test = test["default"]
comparison = reject_inference.compare_models(
    accepted_model, augmented_model, X_test, y_test, feature_cols
)

scores_accepted = accepted_model.predict_proba(X_test)[:, 1]
scores_augmented = augmented_model.predict_proba(X_test)[:, 1]
auc_acc = comparison["champion"]["auc"]
auc_aug = comparison["augmented"]["auc"]

comparison_table = pd.DataFrame(
    [comparison["champion"], comparison["augmented"]],
    index=["Champion", "Augmented"],
)
print("Model Comparison on Hold-Out Accepted Test Set")
print(comparison_table.to_string())
print(f"PSI between models: {comparison['psi_between_models']:.4f}")
print(f"AUC delta: {comparison['auc_delta']:+.4f}")
print(
    "\nCaution: accepted-only outcomes measure augmentation cost, not "
    "performance on rejected applicants."
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fpr_a, tpr_a, _ = roc_curve(y_test, scores_accepted)
fpr_g, tpr_g, _ = roc_curve(y_test, scores_augmented)
axes[0].plot(
    fpr_a,
    tpr_a,
    label=f"Champion (AUC={auc_acc:.4f})",
    color="#3498db",
    linewidth=2,
)
axes[0].plot(
    fpr_g,
    tpr_g,
    label=f"Augmented (AUC={auc_aug:.4f})",
    color="#e74c3c",
    linewidth=2,
)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve Comparison")
axes[0].legend()

axes[1].hist(
    scores_accepted,
    bins=50,
    alpha=0.5,
    density=True,
    label="Champion",
    color="#3498db",
)
axes[1].hist(
    scores_augmented,
    bins=50,
    alpha=0.5,
    density=True,
    label="Augmented",
    color="#e74c3c",
)
axes[1].set_xlabel("Raw Model Score")
axes[1].set_ylabel("Density")
axes[1].set_title("Score Distribution on Test Set")
axes[1].legend()

for scores, name, color in [
    (scores_accepted, "Champion", "#3498db"),
    (scores_augmented, "Augmented", "#e74c3c"),
]:
    prob_true, prob_pred = calibration_curve(y_test, scores, n_bins=10)
    axes[2].plot(prob_pred, prob_true, "s-", color=color, label=name, linewidth=2)
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[2].set_xlabel("Raw Model Score")
axes[2].set_ylabel("Actual Default Rate")
axes[2].set_title("Uncalibrated Score Reliability")
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Decile Analysis — Rank Ordering Check

In [ ]:
# Decile analysis for both models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, scores, name in [(axes[0], scores_accepted, 'Accepted-Only'),
                           (axes[1], scores_augmented, 'Augmented')]:
    decile_df = pd.DataFrame({'score': scores, 'default': y_test.values})
    # Ranking first guarantees ten bins even when model scores are tied.
    decile_df['decile'] = pd.qcut(
        decile_df['score'].rank(method='first'), 10, labels=range(1, 11)
    )
    
    stats = decile_df.groupby('decile', observed=True).agg(
        default_rate=('default', 'mean'),
        count=('default', 'size'),
    ).reset_index()
    
    bars = ax.bar(stats['decile'].astype(str), stats['default_rate'],
                  color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 10)), edgecolor='black')
    for bar, rate in zip(bars, stats['default_rate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{rate:.1%}', ha='center', fontsize=8)
    ax.set_xlabel('Score Decile (1=lowest risk)')
    ax.set_ylabel('Default Rate')
    ax.set_title(f'{name} Model — Decile Default Rates')
    
    # Check rank ordering
    rates = stats['default_rate'].values
    breaks = sum(1 for i in range(1, len(rates)) if rates[i] < rates[i-1])
    ax.text(0.02, 0.95, f'Rank-order breaks: {breaks}', transform=ax.transAxes,
            fontweight='bold', fontsize=10, va='top')

plt.tight_layout()
plt.show()

## 6. Feature Importance Shift — What Changed?

In [ ]:
# Compare permutation importances (more reliable than tree-based importance for comparison)

# Use a subset for speed
np.random.seed(42)
idx = np.random.choice(len(X_test), size=min(5000, len(X_test)), replace=False)
X_sub = X_test.iloc[idx]
y_sub = y_test.iloc[idx]

perm_acc = permutation_importance(accepted_model, X_sub, y_sub, n_repeats=5, 
                                   scoring='roc_auc', random_state=42, n_jobs=1)
perm_aug = permutation_importance(augmented_model, X_sub, y_sub, n_repeats=5,
                                   scoring='roc_auc', random_state=42, n_jobs=1)

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'accepted_only': perm_acc.importances_mean,
    'augmented': perm_aug.importances_mean,
})
imp_df['shift'] = imp_df['augmented'] - imp_df['accepted_only']
imp_df = imp_df.sort_values('accepted_only', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(imp_df))
width = 0.35
ax.barh(x + width/2, imp_df['accepted_only'], width, label='Accepted-Only', color='#3498db', edgecolor='black')
ax.barh(x - width/2, imp_df['augmented'], width, label='Augmented', color='#e74c3c', edgecolor='black')
ax.set_yticks(x)
ax.set_yticklabels(imp_df['feature'])
ax.set_xlabel('Permutation Importance (AUC drop)')
ax.set_title('Feature Importance Shift After Reject Inference')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Show biggest shifts
print("Biggest importance shifts (augmented - accepted):")
shift_df = pd.DataFrame({
    'feature': feature_cols,
    'accepted_only': perm_acc.importances_mean,
    'augmented': perm_aug.importances_mean,
})
shift_df['shift'] = shift_df['augmented'] - shift_df['accepted_only']
baseline_magnitude = shift_df['accepted_only'].abs()
shift_df['relative_shift_pct'] = np.where(
    baseline_magnitude > 1e-4,
    shift_df['shift'] / baseline_magnitude * 100,
    np.nan,
)
print(shift_df.sort_values('shift', key=abs, ascending=False).head(10).to_string(index=False))

## 7. PSI Between Models — Score Stability

Check if the augmented model produces a fundamentally different score distribution (Population Stability Index).

In [ ]:
def compute_psi(expected, actual, bins=10):
    """Population Stability Index between two score distributions."""
    bin_edges = np.linspace(0, 1, bins + 1)
    exp_counts, _ = np.histogram(expected, bins=bin_edges)
    act_counts, _ = np.histogram(actual, bins=bin_edges)
    
    exp_pct = exp_counts / exp_counts.sum()
    act_pct = act_counts / act_counts.sum()
    
    # Avoid log(0)
    exp_pct = np.clip(exp_pct, 1e-6, None)
    act_pct = np.clip(act_pct, 1e-6, None)
    
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi, bin_edges, exp_pct, act_pct

psi, edges, pct_acc, pct_aug = compute_psi(scores_accepted, scores_augmented, bins=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSI bar comparison
midpoints = (edges[:-1] + edges[1:]) / 2
width = (edges[1] - edges[0]) * 0.4
axes[0].bar(midpoints - width/2, pct_acc, width, label='Accepted-Only', color='#3498db', edgecolor='black')
axes[0].bar(midpoints + width/2, pct_aug, width, label='Augmented', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Score Bin')
axes[0].set_ylabel('Proportion')
axes[0].set_title(f'Score Distribution Comparison (PSI = {psi:.4f})')
axes[0].legend()

# Per-bin PSI contribution
per_bin_psi = (pct_aug - pct_acc) * np.log(pct_aug / pct_acc)
colors = ['#e74c3c' if p > 0.02 else '#f39c12' if p > 0.01 else '#2ecc71' for p in per_bin_psi]
axes[1].bar(range(1, 11), per_bin_psi, color=colors, edgecolor='black')
axes[1].set_xlabel('Score Bin')
axes[1].set_ylabel('PSI Contribution')
axes[1].set_title('Per-Bin PSI Contribution')
axes[1].axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"Overall PSI: {psi:.4f}")
if psi < 0.10:
    print("Interpretation: Minimal shift — models produce similar score distributions.")
elif psi < 0.25:
    print("Interpretation: Moderate shift — investigate which score bands changed.")
else:
    print("Interpretation: Significant shift — augmented model behaves very differently.")

## 8. Conclusion

**Key Takeaways:**

1. **Selection bias is plausible, not directly measurable here** — the accepted-only model has never seen outcomes for rejected applicants, so this dataset cannot quantify the blind spot.

2. **Parcelling is a sensitivity analysis, not proof of correction** — pseudo-labels expose the model to a broader feature population, but they encode the champion's ranking and an assumed default rate rather than new evidence.

3. **Sample weighting limits influence** — rejected pseudo-labels are uncertain, so the production helper weights them at 0.3 versus 1.0 for observed outcomes. It also excludes pseudo-labels from calibration.

4. **Production considerations:**
   - More sophisticated methods such as EM or Heckman correction still require defensible assumptions about the selection mechanism
   - The assumed reject default rate of 2x accepted is a heuristic and should be tested with randomized approvals or external outcomes
   - Accepted-only AUC can measure augmentation cost, but not performance on rejected applicants
   - Model-risk documentation should record the assumptions, sensitivity range, and validation evidence